# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MohamedRamadan164/FlyRank_ML_internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MohamedRamadan164/FlyRank_ML_internship"
REPO_DIR = "FlyRank_ML_internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
os.makedirs("work/outputs", exist_ok=True)
print("rows:", df.shape[0])


rows: 30000


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**Two signals, checked first.**

**Signal A — staleness (behind the refresh flags).** Claim: pages that haven't been updated in a long time decline more. Test: decline rate (`trend_direction == "down"`) grouped by `freshness_tier`, with n per bucket.

**Signal B — CTR vs. position (behind the CTR-fix logic).** Claim: pages ranking better get more clicks per impression. Test: weighted CTR (`sum(clicks_90d) / sum(impressions_90d)`, not the mean of per-row CTRs — averaging per-row rates would misweight low-impression rows) grouped by `position_tier`, on rows where `avg_position > 0` (0 means "no data", not rank zero).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# --- Signal A: staleness -> decline rate ---
sigA = df.groupby("freshness_tier").agg(
    n=("content_id", "size"),
    decline_rate=("trend_direction", lambda s: (s == "down").mean()),
).reindex(["0-30", "31-90", "91-180", "181+"])
print("Signal A -- staleness vs decline rate:")
print(sigA.round(3))
print()

# --- Signal B: position -> weighted CTR ---
valid = df[df["avg_position"] > 0].copy()
sigB = valid.groupby("position_tier").apply(
    lambda s: pd.Series({
        "n": len(s),
        "weighted_ctr_pct": 100 * s["clicks_90d"].sum() / s["impressions_90d"].sum(),
    })
).reindex(["top_3", "page_1", "striking", "page_3_5", "deep"])
print("Signal B -- position tier vs weighted CTR:")
print(sigB.round(3))


Signal A -- staleness vs decline rate:
                    n  decline_rate
freshness_tier                     
0-30            20480         0.511
31-90             175         0.589
91-180           9171         0.611
181+              174         0.471

Signal B -- position tier vs weighted CTR:
                     n  weighted_ctr_pct
position_tier                           
top_3           1116.0             0.489
page_1         11814.0             0.350
striking        7304.0             0.347
page_3_5        7242.0             0.155
deep            1319.0             0.041


/tmp/ipykernel_3540/3287785480.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sigB = valid.groupby("position_tier").apply(


**Verdicts.**

- **Signal A (staleness -> decline): MIXED.** Decline rate does *not* rise cleanly with age — it climbs from 0-30 days (51.1%) to 91-180 days (61.1%), but then *drops* at 181+ days (47.1%, n=174, a small bucket). A pure "older = more likely declining" rule would be wrong at the tail. Useful, honest negative: I'll gate on a moderate staleness band (≥90 days) rather than "the older the worse."
- **Signal B (position -> CTR): CONFIRMED.** Weighted CTR drops steadily as position tier worsens (top_3 ≈0.49% → page_1 ≈0.35% → page_3_5 ≈0.15% → deep ≈0.04%; `striking` sits close to `page_1`, ≈0.35%, but the overall gradient holds). Every bucket has n well above the ~50-row floor. This is the real signal behind the CTR-fix logic: a page's own CTR can be compared against what its position tier normally gets.

**The rule, in plain words.** A page is worth a refresh review if it's underperforming CTR for its own position tier (the CTR-fix signal, confirmed), it hasn't been touched in at least 90 days (the staleness signal, using the *moderate* band the mixed result actually supports, not an unbounded "older is worse"), and it's still pulling real impressions (so a fix would matter). Score: `stale_flag × visible_flag × ctr_gap × impressions_90d`. One reason code: `stale_ctr_underperform`. One action label: `review_for_refresh`.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# expected CTR per position tier, from the Signal B benchmark above (weighted, avg_position>0 only)
bench = valid.groupby("position_tier")["clicks_90d"].sum() / valid.groupby("position_tier")["impressions_90d"].sum() * 100
df["expected_ctr"] = df["position_tier"].map(bench)

df["ctr_gap"] = (df["expected_ctr"] - df["ctr"]).clip(lower=0)
df["ctr_gap"] = df["ctr_gap"].where(df["avg_position"] > 0, 0)  # no gap claim without real position data

vis_threshold = df["impressions_90d"].median()
stale_flag = (df["days_since_last_update"] >= 90).astype(int)
visible_flag = (df["impressions_90d"] >= vis_threshold).astype(int)

df["score"] = stale_flag * visible_flag * df["ctr_gap"] * df["impressions_90d"]
df["reason_code"] = "stale_ctr_underperform"
df["action"] = "review_for_refresh"

ranked = df.sort_values("score", ascending=False).reset_index(drop=True)

out_cols = [
    "content_id", "client_id", "score", "reason_code", "action",
    "ctr", "expected_ctr", "ctr_gap", "avg_position", "position_tier",
    "days_since_last_update", "freshness_tier", "impressions_90d",
]
ranked[out_cols].to_csv("work/outputs/baseline_action_score.csv", index=False)

print("vis_threshold (median impressions_90d):", vis_threshold)
print("rows with score > 0:", int((df['score'] > 0).sum()), "of", len(df))
print("wrote work/outputs/baseline_action_score.csv")
ranked[out_cols].head(20)


vis_threshold (median impressions_90d): 731.0
rows with score > 0: 4129 of 30000
wrote work/outputs/baseline_action_score.csv


,content_id,client_id,score,reason_code,action,ctr,expected_ctr,ctr_gap,avg_position,position_tier,days_since_last_update,freshness_tier,impressions_90d
0,content_5fe46e04994d,client_4e07408562,108887.742905,stale_ctr_underperform,review_for_refresh,0.14,0.350324,0.210324,4.2,page_1,104,91-180,517715
1,content_36ff89c8214e,client_19581e27de,88624.627778,stale_ctr_underperform,review_for_refresh,0.05,0.350324,0.300324,7.3,page_1,104,91-180,295097
2,content_c8e9d6ab9013,client_19581e27de,73104.852519,stale_ctr_underperform,review_for_refresh,0.00,0.350324,0.350324,9.7,page_1,104,91-180,208678
3,content_4a6607efcb46,client_6208ef0f77,61286.156110,stale_ctr_underperform,review_for_refresh,0.01,0.488544,0.478544,2.2,top_3,104,91-180,128068
4,content_cb112fce36be,client_19581e27de,58983.222991,stale_ctr_underperform,review_for_refresh,0.16,0.350324,0.190324,5.6,page_1,104,91-180,309910
5,content_a7427266c305,client_19581e27de,48331.742956,stale_ctr_underperform,review_for_refresh,0.11,0.350324,0.240324,5.7,page_1,104,91-180,201111
6,content_91652435f57a,client_19581e27de,46332.761922,stale_ctr_underperform,review_for_refresh,0.06,0.350324,0.290324,7.8,page_1,104,91-180,159590
7,content_c1fe78bc4e37,client_19581e27de,42940.995820,stale_ctr_underperform,review_for_refresh,0.03,0.350324,0.320324,7.5,page_1,104,91-180,134055
8,content_97a86caf3a3d,client_19581e27de,41395.403221,stale_ctr_underperform,review_for_refresh,0.07,0.350324,0.280324,6.4,page_1,104,91-180,147670
9,content_b115f7c74779,client_19581e27de,39550.048957,stale_ctr_underperform,review_for_refresh,0.03,0.350324,0.320324,8.0,page_1,104,91-180,123469


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top10 = ranked[out_cols].head(20)
print(top10.to_string(index=False))


          content_id         client_id         score            reason_code             action  ctr  expected_ctr  ctr_gap  avg_position position_tier  days_since_last_update freshness_tier  impressions_90d
content_5fe46e04994d client_4e07408562 108887.742905 stale_ctr_underperform review_for_refresh 0.14      0.350324 0.210324           4.2        page_1                     104         91-180           517715
content_36ff89c8214e client_19581e27de  88624.627778 stale_ctr_underperform review_for_refresh 0.05      0.350324 0.300324           7.3        page_1                     104         91-180           295097
content_c8e9d6ab9013 client_19581e27de  73104.852519 stale_ctr_underperform review_for_refresh 0.00      0.350324 0.350324           9.7        page_1                     104         91-180           208678
content_4a6607efcb46 client_6208ef0f77  61286.156110 stale_ctr_underperform review_for_refresh 0.01      0.488544 0.478544           2.2         top_3                     1

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weakest pick: #3, `content_c8e9d6ab9013`.** Zero clicks on 208,678 impressions is either a genuinely strong signal or a tracking artifact — and its missing `word_count` (a pattern tied to `content_type == "keyword article"`, not random) means I can't fully trust its other metadata either. I'm keeping it in the queue but flagging it for a manual check before action, not an automatic refresh.

**Second-weakest: #10, `content_b115f7c74779`.** `days_since_last_update = 104` is being read as "stale," but for a page that's simply new, the same number means "published 104 days ago, never touched since" — which looks identical to "was good once, now neglected" in this rule. The rule can't currently tell those apart without a `first_published` vs `last_updated` split.

**Leakage check.** The score uses only: `position_tier`/`avg_position`, `ctr`, `impressions_90d`, `days_since_last_update`, `freshness_tier` — all attributes of the page's *current* state, knowable before any decision. It does **not** use `trend_direction` or `trend_pct` (the week-2/3 label and the value it's derived from), and it does not use any future-window column. Confirmed below by checking the rule's inputs against that list programmatically, not just by eye.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
rule_inputs = {"avg_position", "position_tier", "ctr", "expected_ctr", "impressions_90d",
               "days_since_last_update", "freshness_tier"}
forbidden = {"trend_direction", "trend_pct", "is_declining_label"}

overlap = rule_inputs & forbidden
print("Rule inputs:", sorted(rule_inputs))
print("Forbidden (label-derived / future):", sorted(forbidden))
print("Overlap (should be empty):", overlap)
assert not overlap, "Leakage: a label-derived column made it into the rule."
print("OK -- no label-derived or future-window column in the score.")


Rule inputs: ['avg_position', 'ctr', 'days_since_last_update', 'expected_ctr', 'freshness_tier', 'impressions_90d', 'position_tier']
Forbidden (label-derived / future): ['is_declining_label', 'trend_direction', 'trend_pct']
Overlap (should be empty): set()
OK -- no label-derived or future-window column in the score.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.